# Train Siamese LSTM Truyen Thong ? Triplet Euclidean

Notebook nay dung pipeline `data_ready` giong `train-siamese-bilstm-cosine.ipynb`, nhung encoder la **LSTM 1 chieu (traditional)** va toi uu voi **triplet Euclidean loss** theo baseline paper.


In [24]:
import os
import re
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


Device: cuda
GPU: Tesla T4


In [25]:
def detect_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/hngphtrn/legals/data/data_ready'),
        Path('/kaggle/input/datasets/hngphtrn/legals/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data/data_ready'),
        Path('/kaggle/input/vnlegal-rag/data_ready'),
        Path('/kaggle/working/vnlegal-rag/data/data_ready'),
        Path('/kaggle/working/legals/data/data_ready'),
        Path('/kaggle/working/data/data_ready'),
        Path('data/data_ready'),
        Path('../data/data_ready'),
    ]
    for p in candidates:
        if p.exists() and (p / 'qa_train.csv').exists() and (p / 'corpus_train.csv').exists():
            return p
    raise FileNotFoundError('Khong tim thay data_ready. Hay kiem tra duong dan dataset tren Kaggle.')


DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path('/kaggle/working/siamese_lstm_traditional_artifacts' if Path('/kaggle').exists() else 'artifacts/siamese_lstm_traditional')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)


DATA_DIR = /kaggle/input/datasets/hngphtrn/legals/data_ready
ARTIFACT_DIR = /kaggle/working/siamese_lstm_traditional_artifacts


In [26]:
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='	')
qa_val = pd.read_csv(DATA_DIR / 'qa_val.csv', sep='	')
qa_test = pd.read_csv(DATA_DIR / 'qa_test.csv', sep='	')
corpus_train = pd.read_csv(DATA_DIR / 'corpus_train.csv', sep='	')
corpus_val = pd.read_csv(DATA_DIR / 'corpus_val.csv', sep='	')
corpus_test = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='	')

print('qa_train:', qa_train.shape, '| corpus_train:', corpus_train.shape)
print('qa_val:  ', qa_val.shape, '| corpus_val:  ', corpus_val.shape)
print('qa_test: ', qa_test.shape, '| corpus_test: ', corpus_test.shape)


def pick_first_existing_col(df, candidates, df_name):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Khong tim thay cot nao trong {candidates} cho {df_name}. Cot hien co: {list(df.columns)}")


QA_TEXT_COL = pick_first_existing_col(qa_train, ['question', 'query', 'text', 'question_text'], 'qa_train')
CORPUS_TEXT_COL = pick_first_existing_col(
    corpus_train,
    ['article_chunk', 'article_content', 'context', 'text', 'content', 'chunk_text', 'passage_text'],
    'corpus_train',
)
print('QA_TEXT_COL =', QA_TEXT_COL)
print('CORPUS_TEXT_COL =', CORPUS_TEXT_COL)


qa_train: (23311, 14) | corpus_train: (7771, 6)
qa_val:   (2841, 14) | corpus_val:   (947, 6)
qa_test:  (2991, 14) | corpus_test:  (997, 6)
QA_TEXT_COL = question
CORPUS_TEXT_COL = article_content


In [27]:
try:
    from underthesea import word_tokenize
    USE_UNDERTHESEA = True
except Exception:
    USE_UNDERTHESEA = False
    print('underthesea chua co. Chay cell sau de cai dat neu can.')


def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize(text: str):
    text = normalize_text(text)
    if USE_UNDERTHESEA:
        return word_tokenize(text, format='text').split()
    # Fallback robust on Kaggle/Python 3.12 when underthesea is unavailable
    return re.findall(r"\w+", text, flags=re.UNICODE)


underthesea chua co. Chay cell sau de cai dat neu can.


In [28]:
# Underthesea co the loi build tren Python 3.12 (Kaggle) vi phu thuoc native.
# Neu ban dung Python 3.10/3.11, co the thu:
# !pip -q install underthesea==6.8.4
#
# Neu van loi, giu fallback tokenizer (regex) va bo qua cai dat underthesea.
# Fallback da du de train baseline va so sanh metric.


In [29]:
PAD = '<PAD>'
UNK = '<UNK>'
MAX_VOCAB = 30000
MAX_Q_LEN = 64
MAX_D_LEN = 256

counter = Counter()
for txt in qa_train[QA_TEXT_COL].astype(str).tolist():
    counter.update(tokenize(txt))
for txt in corpus_train[CORPUS_TEXT_COL].astype(str).tolist():
    counter.update(tokenize(txt))

most_common = counter.most_common(MAX_VOCAB - 2)
itos = {0: PAD, 1: UNK}
for i, (tok, _) in enumerate(most_common, start=2):
    itos[i] = tok
stoi = {tok: idx for idx, tok in itos.items()}


def encode_text(text, max_len):
    toks = tokenize(text)
    ids = [stoi.get(t, stoi[UNK]) for t in toks[:max_len]]
    if len(ids) < max_len:
        ids += [stoi[PAD]] * (max_len - len(ids))
    return ids

print('Vocab size:', len(stoi))


Vocab size: 5800


In [30]:
corpus_train_map = corpus_train.set_index('passage_id').to_dict(orient='index')
corpus_val_map = corpus_val.set_index('passage_id').to_dict(orient='index')
corpus_test_map = corpus_test.set_index('passage_id').to_dict(orient='index')

domain_to_passages = defaultdict(list)
if 'macro_domain' in corpus_train.columns:
    for _, row in corpus_train.iterrows():
        domain_to_passages[str(row.get('macro_domain', 'unknown'))].append(row['passage_id'])


In [31]:
class TripletQADataset(Dataset):
    def __init__(self, qa_df, corpus_map, domain_to_passages):
        self.qa_df = qa_df.reset_index(drop=True)
        self.corpus_map = corpus_map
        self.domain_to_passages = domain_to_passages

    def __len__(self):
        return len(self.qa_df)

    def _sample_negative(self, pos_pid, macro_domain):
        if self.domain_to_passages and macro_domain in self.domain_to_passages:
            cands = [p for p in self.domain_to_passages[macro_domain] if p != pos_pid]
            if cands:
                return random.choice(cands)
        keys = list(self.corpus_map.keys())
        neg_pid = random.choice(keys)
        while neg_pid == pos_pid:
            neg_pid = random.choice(keys)
        return neg_pid

    def __getitem__(self, idx):
        row = self.qa_df.iloc[idx]
        q = str(row[QA_TEXT_COL])
        pos_pid = row['passage_id']
        macro = str(row.get('macro_domain', 'unknown'))

        pos_doc = self.corpus_map[pos_pid]
        neg_pid = self._sample_negative(pos_pid, macro)
        neg_doc = self.corpus_map[neg_pid]

        return {
            'anchor': torch.tensor(encode_text(q, MAX_Q_LEN), dtype=torch.long),
            'positive': torch.tensor(encode_text(str(pos_doc[CORPUS_TEXT_COL]), MAX_D_LEN), dtype=torch.long),
            'negative': torch.tensor(encode_text(str(neg_doc[CORPUS_TEXT_COL]), MAX_D_LEN), dtype=torch.long),
        }


In [32]:
train_ds = TripletQADataset(qa_train, corpus_train_map, domain_to_passages)

NUM_WORKERS = 0
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)
print('Train triplets:', len(train_ds))


Train triplets: 23311


In [33]:
class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=256, num_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.pad_idx = pad_idx

    def forward(self, x):
        mask = (x != self.pad_idx).float().unsqueeze(-1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        summed = (out * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        pooled = summed / denom
        return F.normalize(pooled, p=2, dim=1)


class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=200, hidden_size=256, num_layers=1, dropout=0.3, pad_idx=0):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_size, num_layers, dropout, pad_idx)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, a, p, n):
        return self.encode(a), self.encode(p), self.encode(n)


In [34]:
model = SiameseLSTM(vocab_size=len(stoi), pad_idx=stoi['<PAD>']).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())
margin = 0.75


def triplet_loss_euclidean(a, p, n, margin=0.75):
    d_ap = torch.norm(a - p, p=2, dim=1)
    d_an = torch.norm(a - n, p=2, dim=1)
    return F.relu(margin + d_ap - d_an).mean()


In [35]:
def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    for batch in tqdm(loader, leave=False):
        a = batch['anchor'].to(device)
        p = batch['positive'].to(device)
        n = batch['negative'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            za, zp, zn = model(a, p, n)
            loss = triplet_loss_euclidean(za, zp, zn, margin=margin)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total += float(loss.item())
    return total / max(1, len(loader))


In [36]:
@torch.no_grad()
def encode_passage_matrix(model, corpus_df, batch_size=64):
    model.eval()
    pids = corpus_df['passage_id'].tolist()
    vecs = []
    for i in range(0, len(pids), batch_size):
        chunk = pids[i:i + batch_size]
        texts = corpus_df.iloc[i:i + batch_size][CORPUS_TEXT_COL].astype(str).tolist()
        ids = torch.tensor([encode_text(t, MAX_D_LEN) for t in texts], dtype=torch.long, device=device)
        z = model.encode(ids)
        vecs.append(z.cpu())
    return pids, torch.cat(vecs, dim=0)


@torch.no_grad()
def evaluate_retrieval(model, qa_df, corpus_df, topk=(1, 3, 5), max_queries=1500):
    model.eval()
    pids, pmat = encode_passage_matrix(model, corpus_df, batch_size=64)
    pid_to_idx = {p: i for i, p in enumerate(pids)}

    qa_eval = qa_df if (max_queries is None or len(qa_df) <= max_queries) else qa_df.sample(max_queries, random_state=42)

    hits = {k: 0 for k in topk}
    rr_sum = 0.0

    for _, row in tqdm(qa_eval.iterrows(), total=len(qa_eval), leave=False):
        q_ids = torch.tensor([encode_text(str(row[QA_TEXT_COL]), MAX_Q_LEN)], dtype=torch.long, device=device)
        q_vec = model.encode(q_ids).cpu()

        dists = torch.cdist(q_vec, pmat, p=2).squeeze(0)
        order = torch.argsort(dists, descending=False)

        true_pid = row['passage_id']
        true_idx = pid_to_idx.get(true_pid, None)
        if true_idx is None:
            continue

        rank_pos = (order == true_idx).nonzero(as_tuple=True)[0]
        if len(rank_pos) == 0:
            continue
        rank = int(rank_pos.item()) + 1
        rr_sum += 1.0 / rank

        for k in topk:
            if rank <= k:
                hits[k] += 1

    n = len(qa_eval)
    metrics = {f'Recall@{k}': hits[k] / max(1, n) for k in topk}
    metrics['MRR'] = rr_sum / max(1, n)
    return metrics


In [37]:
EPOCHS = 20
PATIENCE = 5
MIN_DELTA = 1e-4
best_score = -1.0
best_epoch = 0
wait = 0
best_path = ARTIFACT_DIR / 'siamese_lstm_traditional_best.pt'
history = []

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader)
    val_metrics = evaluate_retrieval(model, qa_val, corpus_val, topk=(1, 3, 5), max_queries=1500)
    val_score = val_metrics['MRR']
    scheduler.step(val_score)

    row = {'epoch': epoch, 'train_loss': tr_loss, **val_metrics}
    history.append(row)
    print(
        f"Epoch {epoch:02d} | loss={tr_loss:.4f} | MRR={val_metrics['MRR']:.4f} | "
        f"R@1={val_metrics['Recall@1']:.4f} | R@5={val_metrics['Recall@5']:.4f}"
    )

    if val_score > best_score + MIN_DELTA:
        best_score = val_score
        best_epoch = epoch
        wait = 0
        torch.save(model.state_dict(), best_path)
        print(f"  -> Saved best checkpoint at epoch {epoch:02d} (MRR={best_score:.4f})")
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch:02d}. Best epoch: {best_epoch:02d} (MRR={best_score:.4f})")
            break

print(f"Best val MRR: {best_score:.4f} (epoch {best_epoch:02d})")
print(f"Best checkpoint: {best_path}")


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 01 | loss=0.2112 | MRR=0.5577 | R@1=0.4487 | R@5=0.6840
  -> Saved best checkpoint at epoch 01 (MRR=0.5577)


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

...

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 17 | loss=0.0995 | MRR=0.5657 | R@1=0.4493 | R@5=0.7113


  0%|          | 0/729 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

Epoch 18 | loss=0.1011 | MRR=0.5647 | R@1=0.4467 | R@5=0.7107
Early stopping triggered at epoch 18. Best epoch: 13 (MRR=0.5680)
Best val MRR: 0.5680 (epoch 13)
Best checkpoint: /kaggle/working/siamese_lstm_traditional_artifacts/siamese_lstm_traditional_best.pt


In [38]:
model.load_state_dict(torch.load(best_path, map_location=device))
test_metrics = evaluate_retrieval(model, qa_test, corpus_test, topk=(1, 3, 5, 10), max_queries=None)
print('Test metrics:', test_metrics)


  0%|          | 0/2991 [00:00<?, ?it/s]

Test metrics: {'Recall@1': 0.458040789033768, 'Recall@3': 0.636242059511869, 'Recall@5': 0.7041123370110332, 'Recall@10': 0.7873620862587764, 'MRR': 0.5722939746034852}


In [39]:
with open(ARTIFACT_DIR / 'tokenizer_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({'stoi': stoi, 'itos': {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)

with open(ARTIFACT_DIR / 'siamese_meta.json', 'w', encoding='utf-8') as f:
    json.dump(
        {
            'distance_metric': 'euclidean',
            'encoder': 'lstm_traditional',
            'max_q_len': MAX_Q_LEN,
            'max_d_len': MAX_D_LEN,
            'margin': margin,
            'embed_dim': 200,
            'hidden_size': 256,
            'num_layers': 1,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame(history).to_csv(ARTIFACT_DIR / 'train_history.csv', index=False)
print('Saved artifacts at:', ARTIFACT_DIR)


Saved artifacts at: /kaggle/working/siamese_lstm_traditional_artifacts
